In [2]:
import os
import time
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from pathlib import Path
from dotenv import load_dotenv
import os
from pathlib import Path
DATA_DIR = str(Path.cwd().parent / "데이터수집" / "data")
print("기준 폴더:", DATA_DIR)

#import httpx
# =========================
# OpenAI 설정
# =========================
env_path = Path.cwd().parent / ".env"   # 회차별_회의록_분석 → 코드/.env
load_dotenv(env_path)

API_KEY = os.getenv("OPENAI_API_KEY")
if not API_KEY:
    raise ValueError("❌ OPENAI_API_KEY를 .env에서 불러오지 못했습니다.")

client = OpenAI(
    api_key=API_KEY
    # http_client=httpx.Client(verify=False, timeout=120), 내부망일경우 SSL 인증 문제로 인해 추가 설정 필요
)  # OPENAI_API_KEY 환경변수에 세팅되어 있어야 함
MODEL_NAME = "gpt-4.1-mini"  # 필요시 변경

# =========================
# 1) 요약 프롬프트
# =========================
SYSTEM_PROMPT_SUMMARY = """
너는 국회 회의록 요약 전문가다.
발언자의 발언 전체를 바탕으로 핵심만 요약한다.

요약 규칙:
- 3줄 이내
- 쟁점, 문제제기, 설명, 답변, 요청사항 중심
- 불필요한 반복 제거
- 문장은 자연스럽게 서술형으로 작성
"""

def build_summary_prompt(meeting_round, speaker_name, speeches):
    return f"""
다음은 {meeting_round} 회의에서 발언자 '{speaker_name}'의 발언 모음이다.

발언:
\"\"\"
{speeches}
\"\"\"

위 발언을 종합해 3줄 이내로 요약하라.
"""

def summarize_speeches(meeting_round, speaker_name, speeches, max_chars=8000):
    speeches = (speeches or "").strip()
    if not speeches:
        return ""

    if len(speeches) > max_chars:
        speeches = speeches[:max_chars//2] + "\n...\n" + speeches[-max_chars//2:]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_SUMMARY},
            {"role": "user", "content": build_summary_prompt(meeting_round, speaker_name, speeches)}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()


# =========================
# 2) 소속·직위 처리
# =========================
OFFICIAL_ROLE_SUFFIXES = [
    "사무총장", "차관", "장관", "청장", "본부장",
    "국장", "과장", "담당관", "감사관",
    "위원장", "위원"
]

def split_org_and_title(position):
    position = (position or "").strip()
    if not position:
        return "", ""

    if position in ["위원", "위원장", "의원"]:
        return "", position

    for suf in OFFICIAL_ROLE_SUFFIXES:
        if position.endswith(suf) and len(position) > len(suf):
            return position[:-len(suf)].strip(), suf

    return "", position

def infer_speaker_type(party, position):
    party = (party or "").strip()
    position = (position or "").strip()

    if party and party != "미분류":
        return "lawmaker"
    if "증인" in position or "참고인" in position:
        return "witness"
    if any(position.endswith(s) for s in OFFICIAL_ROLE_SUFFIXES):
        return "government"
    return "unknown"

def enrich_affiliation_columns(df, committee_name="행정안전위원회"):
    df = df.copy()

    df["speaker_name"] = df["speaker_name"].astype(str).str.strip()
    df["speaker_position"] = df["speaker_position"].fillna("").astype(str).str.strip()
    df["party"] = df["party"].fillna("").astype(str).str.strip()
    df["speech_text"] = df["speech_text"].fillna("").astype(str)

    df["speaker_type"] = df.apply(
        lambda r: infer_speaker_type(r["party"], r["speaker_position"]),
        axis=1
    )

    org_title = df["speaker_position"].apply(split_org_and_title)
    df["organization"] = [o for o, t in org_title]
    df["position_title"] = [t for o, t in org_title]

    df["party_clean"] = df.apply(
        lambda r: r["party"] if r["speaker_type"] == "lawmaker" and r["party"] != "미분류" else "",
        axis=1
    )

    df["organization"] = df.apply(
        lambda r: f"국회 {committee_name}" if r["speaker_type"] == "lawmaker" else r["organization"],
        axis=1
    )

    # 🔹 소속_직위 컬럼
    df["소속_직위"] = df.apply(
        lambda r: "" if r["speaker_type"] == "lawmaker"
        else " ".join(p for p in [r["organization"], r["position_title"]] if p),
        axis=1
    )

    df["speaker_key"] = df.apply(
        lambda r: f"{r['speaker_name']}|{r['speaker_type']}|{r['party_clean']}|{r['소속_직위']}",
        axis=1
    )

    return df

def group_speeches(df):
    return (
        df.groupby(
            ["speaker_key", "speaker_name", "speaker_type",
             "party_clean", "organization", "position_title", "소속_직위"]
        )["speech_text"]
        .apply(lambda x: " ".join(x))
        .reset_index(name="speeches")
    )


# =========================
# 3) 요약 문장 접두어 생성 (🔥 핵심)
# =========================
def build_summary_prefix(row):
    name = row["speaker_name"]
    party = row["party_clean"]
    affiliation = row["소속_직위"]
    speaker_type = row["speaker_type"]

    if speaker_type == "lawmaker":
        if party:
            return f"{name} 의원({party})은 "
        else:
            return f"{name} 의원({affiliation})은 "

    if affiliation:
        return f"{name}({affiliation})은 "

    return f"{name}은 "


# =========================
# 4) 메인 파이프라인
# =========================
def run_pipeline():
    results = []

    folders = sorted(
        [f for f in os.listdir(DATA_DIR) if f.startswith("제") and f.endswith("회")],
        key=lambda x: int(x.replace("제", "").replace("회", ""))
    )

    for folder in folders:
        meeting_round = folder.replace("제", "").replace("회", "") + "회"
        folder_path = os.path.join(DATA_DIR, folder)
        print(f"\n▶ 처리 중: {meeting_round}")

        for file in os.listdir(folder_path):
            if not file.endswith("_minutes_speeches.csv"):
                continue

            df = pd.read_csv(os.path.join(folder_path, file))
            df = enrich_affiliation_columns(df)
            grouped = group_speeches(df)

            for _, row in tqdm(grouped.iterrows(), total=len(grouped)):
                raw_summary = summarize_speeches(
                    meeting_round,
                    row["speaker_name"],
                    row["speeches"]
                )

                prefix = build_summary_prefix(row)
                summary = prefix + raw_summary

                results.append({
                    "회차": meeting_round,
                    "발언자명": row["speaker_name"],
                    "발언자유형": row["speaker_type"],
                    "정당": row["party_clean"],
                    "소속기관": row["organization"],
                    "직위": row["position_title"],
                    "소속_직위": row["소속_직위"],
                    "발화내용 요약": summary
                })

                time.sleep(0.25)

    return pd.DataFrame(results)


# =========================
# 5) 실행
# =========================
summary_df = run_pipeline()

output_path = os.path.join(
    DATA_DIR,
    "회차별_발언자별_발화요약_소속_직위_포함.csv"
)
summary_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("✅ 저장 완료:", output_path)
summary_df.head()



▶ 처리 중: 415회


100%|██████████████████████████████████████████████████████████████████████████████████| 29/29 [01:05<00:00,  2.24s/it]



▶ 처리 중: 416회


100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.81s/it]



▶ 처리 중: 417회


...

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.97s/it]



▶ 처리 중: 430회


100%|██████████████████████████████████████████████████████████████████████████████████| 19/19 [00:41<00:00,  2.19s/it]

✅ 저장 완료: C:\Users\User\Desktop\data_collection\data\회차별_발언자별_발화요약_소속_직위_포함.csv


,회차,발언자명,발언자유형,정당,소속기관,직위,소속_직위,발화내용 요약
0,415회,김성회,lawmaker,더불어민주당,국회 행정안전위원회,위원,,김성회 의원(더불어민주당)은 김성회 위원은 행정안전위원회에 다시 합류한 소감을 밝히...
1,415회,모경종,lawmaker,더불어민주당,국회 행정안전위원회,위원,,"모경종 의원(더불어민주당)은 모경종 의원은 행정안전위원회의 중요성을 강조하며, 총선..."
2,415회,박정현,lawmaker,더불어민주당,국회 행정안전위원회,위원,,박정현 의원(더불어민주당)은 박정현 의원은 국민의힘 의원들이 총선 민의를 무시하고 ...
3,415회,신정훈,lawmaker,더불어민주당,국회 행정안전위원회,위원장,,신정훈 의원(더불어민주당)은 신정훈 위원장은 제22대 행정안전위원회 위원장으로서 지...
4,415회,양부남,lawmaker,더불어민주당,국회 행정안전위원회,위원,,양부남 의원(더불어민주당)은 양부남 위원은 국민의 생명과 안전을 최우선으로 보호해야...


In [9]:
#import httpx
# =========================
# OpenAI 설정
# =========================
API_KEY = '' # API 하드코딩용
client = OpenAI(
    api_key=API_KEY
    # http_client=httpx.Client(verify=False, timeout=120), 내부망일경우 SSL 인증 문제로 인해 추가 설정 필요
)  # OPENAI_API_KEY 환경변수에 세팅되어 있어야 함
MODEL_NAME = "gpt-4.1-mini"  # 필요시 변경

In [11]:
SYSTEM_PROMPT_SUMMARY = """
너는 국회 회의록 요약 전문가다.
한 회차에서 특정 발언자(위원/의원/정부기관 관계자 등)의 발언 전체를 바탕으로 핵심만 요약한다.

요약 규칙:
- 3줄 이내
- 쟁점/문제제기/설명/답변/요청사항 중심
- 불필요한 수식/반복 제거
- 가능한 경우 핵심 키워드를 포함
"""


In [12]:
def build_summary_prompt(meeting_round: str, speaker_name: str, speeches: str) -> str:
    return f"""
다음은 {meeting_round} 회의에서 발언자 '{speaker_name}'의 발언 모음이다.

발언:
\"\"\"
{speeches}
\"\"\"

위 발언을 종합해 3줄 이내로 요약하라.
"""



In [13]:
def pick_first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """
    df에 존재하는 컬럼들 중 candidates 순서대로 처음 발견되는 컬럼명을 반환
    """
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def normalize_speaker_meta(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """
    다양한 회의록 CSV 스키마를 최대한 흡수하기 위한 정규화.
    반환:
      - 정규화된 df (speaker_name, speech_text, party_like 컬럼을 보장하려 시도)
      - 어떤 컬럼을 썼는지 메타 정보(dict)
    """
    meta = {}

    # 발언자명 후보 컬럼들 (필요에 따라 추가)
    speaker_col = pick_first_existing_column(df, [
        "speaker_name", "의원명", "발언자", "발언자명", "name"
    ])
    if speaker_col is None:
        raise ValueError("발언자명 컬럼을 찾지 못했습니다. (speaker_name/의원명/발언자 등 확인 필요)")
    meta["speaker_col"] = speaker_col

    # 발언 텍스트 후보 컬럼들
    speech_col = pick_first_existing_column(df, [
        "speech_text", "발언내용", "발화내용", "text", "content"
    ])
    if speech_col is None:
        raise ValueError("발언 텍스트 컬럼을 찾지 못했습니다. (speech_text/발언내용 등 확인 필요)")
    meta["speech_col"] = speech_col

    # 정당/소속 후보 컬럼들
    party_col = pick_first_existing_column(df, [
        "party", "정당", "정당명"
    ])
    # 정부/기관 관계자용 소속/기관/직위 후보 (정당이 비었을 때 대체)
    org_col = pick_first_existing_column(df, [
        "organization", "기관", "소속", "부처", "직위", "부서", "소속기관", "기관명"
    ])

    meta["party_col"] = party_col
    meta["org_col"] = org_col

    # 정규화된 표준 컬럼 생성
    out = df.copy()
    out["__speaker_name"] = out[speaker_col].astype(str)
    out["__speech_text"] = out[speech_col].fillna("").astype(str)

    # "정당" 칼럼을 최종 출력 컬럼로 유지하되,
    # 정당 컬럼이 없거나 값이 비면 기관/소속/직위 등을 넣어준다.
    if party_col is not None:
        out["__party_like"] = out[party_col].fillna("").astype(str)
    else:
        out["__party_like"] = ""

    if org_col is not None:
        org_val = out[org_col].fillna("").astype(str)
        # 정당이 비어있으면 org_val로 대체
        out["__party_like"] = out["__party_like"].where(out["__party_like"].str.strip() != "", org_val)

    # 그래도 비어있으면 "미상"으로
    out["__party_like"] = out["__party_like"].where(out["__party_like"].str.strip() != "", "미상")

    return out, meta


In [14]:
def summarize_speeches(meeting_round: str, speaker_name: str, speeches: str,
                       model: str = "gpt-4.1-mini",
                       max_chars: int = 8000) -> str:
    """
    회차/발언자별 발언 모음을 3줄 이내로 요약.
    max_chars: 모델에 보내는 텍스트 길이 제한(비용/속도 절감)
    """
    # 너무 길면 앞부분/뒷부분을 섞어 보존(중요 결론이 후반에 나오는 경우 대비)
    speeches = speeches.strip()
    if len(speeches) > max_chars:
        head = speeches[: max_chars // 2]
        tail = speeches[-max_chars // 2 :]
        speeches = head + "\n...\n" + tail

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT_SUMMARY},
                {"role": "user", "content": build_summary_prompt(meeting_round, speaker_name, speeches)}
            ],
            temperature=0.3
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"[요약 실패] {meeting_round} / {speaker_name} :", e)
        return ""


In [15]:
def extract_meeting_round_from_folder(folder_name: str) -> str:
    """
    '제421회' -> '421회'
    """
    return folder_name.replace("제", "").replace("회", "") + "회"


def run_summary_pipeline(data_dir: str,
                         file_suffix: str = "_minutes_speeches.csv",
                         model: str = "gpt-4.1-mini",
                         sleep_sec: float = 0.25) -> pd.DataFrame:
    """
    data_dir 하위의 '제415회'~ 폴더들을 순회하며,
    모든 발언자(위원/의원/정부기관 관계자 등)의 회차별 발언을 합쳐 3줄 요약한다.

    최종 컬럼: 회차, 정당, 의원명(=발언자명), 발화내용 요약
    """
    results = []
    cache = {}  # (회차, 발언자명, 정당/소속) -> 요약 캐시

    folders = [f for f in os.listdir(data_dir) if f.startswith("제") and f.endswith("회")]
    folders.sort(key=lambda x: int(x.replace("제", "").replace("회", "")))  # 회차 순 정렬

    for folder in folders:
        meeting_round = extract_meeting_round_from_folder(folder)
        folder_path = os.path.join(data_dir, folder)
        print(f"\n▶ 처리 중: {meeting_round} ({folder_path})")

        # 해당 폴더 안 CSV들(예: 제1호, 제2호...) 전부 처리
        files = [fn for fn in os.listdir(folder_path) if fn.endswith(file_suffix)]
        files.sort()

        for fn in files:
            file_path = os.path.join(folder_path, fn)
            df = pd.read_csv(file_path)

            # 컬럼 정규화
            norm_df, meta = normalize_speaker_meta(df)

            # (발언자명, 정당/소속) 기준으로 발언 합치기
            grouped = (
                norm_df.groupby(["__party_like", "__speaker_name"])["__speech_text"]
                .apply(lambda x: " ".join([t for t in x.tolist() if isinstance(t, str) and t.strip()]))
                .reset_index()
            )

            for _, row in tqdm(grouped.iterrows(), total=len(grouped)):
                party_like = row["__party_like"]
                speaker_name = row["__speaker_name"]
                speeches = row["__speech_text"]

                if not isinstance(speeches, str) or not speeches.strip():
                    continue

                key = (meeting_round, speaker_name, party_like)
                if key in cache:
                    summary = cache[key]
                else:
                    summary = summarize_speeches(
                        meeting_round=meeting_round,
                        speaker_name=speaker_name,
                        speeches=speeches,
                        model=model
                    )
                    cache[key] = summary
                    time.sleep(sleep_sec)

                results.append({
                    "회차": meeting_round,
                    "정당": party_like,          # 정당 없으면 기관/소속/직위 등이 들어갈 수 있음
                    "의원명": speaker_name,      # 컬럼명은 유지하되 실제로는 '발언자명'
                    "발화내용 요약": summary
                })

    return pd.DataFrame(results)


In [16]:
summary_df = run_summary_pipeline(DATA_DIR, model="gpt-4.1-mini", sleep_sec=0.25)
summary_df.head()



▶ 처리 중: 415회 (C:\Users\User\Desktop\data_collection\data\제415회)


100%|██████████████████████████████████████████████████████████████████████████████████| 29/29 [00:33<00:00,  1.16s/it]



▶ 처리 중: 416회 (C:\Users\User\Desktop\data_collection\data\제416회)


100%|██████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 7300.79it/s]



▶ 처리 중: 417회 (C:\Users\User\Desktop\data_collection\data\제417회)


...


▶ 처리 중: 429회 (C:\Users\User\Desktop\data_collection\data\제429회)


100%|██████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 3194.44it/s]



▶ 처리 중: 430회 (C:\Users\User\Desktop\data_collection\data\제430회)


100%|██████████████████████████████████████████████████████████████████████████████████| 19/19 [00:40<00:00,  2.13s/it]


,회차,정당,의원명,발화내용 요약
0,415회,기본소득당,용혜인,용혜인 의원은 국민의힘의 발목잡기에도 국회가 법과 원칙에 따라 운영되어야 한다고 강...
1,415회,더불어민주당,김성회,김성회 위원은 행안위 보좌진 경험을 바탕으로 다시 위원회에 참여하게 되어 책임감을 ...
2,415회,더불어민주당,모경종,"모경종 의원은 행정안전위원회의 중요성을 강조하며, 총선 후 일부 의원들의 빠른 복귀..."
3,415회,더불어민주당,박정현,박정현 의원은 국민의힘 의원들이 총선 민의를 무시하고 국회에 불참하는 점을 비판하며...
4,415회,더불어민주당,신정훈,신정훈 위원장은 제22대 행정안전위원회 위원장으로서 지방 소멸 위기 극복과 국민 안...


In [17]:
output_path = os.path.join(DATA_DIR, "회차별_발언자별_발화요약_415_430회_통합.csv")
summary_df.to_csv(output_path, index=False, encoding="utf-8-sig")
output_path


'C:\\Users\\User\\Desktop\\data_collection\\data\\회차별_발언자별_발화요약_415_430회_통합.csv'

In [18]:
# CSV 파일 경로
file_path = r"C:\Users\User\Desktop\data_collection\data\회차별_발언자별_발화요약_415_430회_통합.csv"

# CSV 불러오기
df = pd.read_csv(file_path)

# 전체 행 개수 출력
print("전체 행 개수:", len(df))

전체 행 개수: 1012


In [20]:
df = pd.read_csv(file_path)

# ✅ 422회차만 필터링
df_422 = df[df["회차"] == "427회"]

# 결과 확인
print("422회차 요구자료 개수:", len(df_422))
df_422.head()

422회차 요구자료 개수: 62


,회차,정당,의원명,발화내용 요약
689,427회,국민의힘,고동진,"고동진 위원은 부정유통 단속 실태와 진화위 인력 운영의 효율성 문제를 지적하며, 정..."
690,427회,국민의힘,박수민,"박수민 의원은 예산편성 의무 조항이 경직성과 권위 저하를 초래하므로 반대하며, 지역..."
691,427회,국민의힘,이달희,"이달희 위원은 지방자치권 침해와 정부 재정권 제한 문제를 지적하며, 지역사랑상품권 ..."
692,427회,국민의힘,이성권,이성권 의원은 지방자치권 침해 우려와 지방 재정 부담 증가 가능성 때문에 임의규정을...
693,427회,더불어민주당,모경종,모경종 의원은 ‘재정적 지원 의무’ 규정이 다른 법률에도 존재하며 정부 재량권이 중...


In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\User\Desktop\data_collection\data\회차별_발언자별_발화요약_소속_직위_포함.csv")

dup_by_round = df[df.duplicated(subset=["회차"], keep=False)]

print(f"회차 기준 중복 행 수: {len(dup_by_round)}")
dup_by_round.sort_values("회차").head(10)


회차 기준 중복 행 수: 1051


,회차,발언자명,발언자유형,정당,소속기관,직위,소속_직위,발화내용 요약
0,415회,김성회,lawmaker,더불어민주당,국회 행정안전위원회,위원,NaN,김성회 의원(더불어민주당)은 김성회 위원은 행정안전위원회에 다시 합류한 소감을 밝히...
34,415회,김상욱,lawmaker,더불어민주당,국회 행정안전위원회,위원,NaN,김상욱 의원(더불어민주당)은 김상욱 의원은 화성 아리셀 공장 화재를 계기로 형식적인...
35,415회,김성회,lawmaker,더불어민주당,국회 행정안전위원회,위원,NaN,김성회 의원(더불어민주당)은 김성회 의원은 소방활동 정보카드의 부실함과 인력 부족 ...
36,415회,김종양,lawmaker,국민의힘,국회 행정안전위원회,위원,NaN,"김종양 의원(국민의힘)은 김종양 의원은 다수당의 입법 횡포에 대한 사과를 요구하며,..."
37,415회,모경종,lawmaker,더불어민주당,국회 행정안전위원회,위원,NaN,모경종 의원(더불어민주당)은 모경종 의원은 소방청과 행안부의 일부 핵심 인사들이 업...
38,415회,박정현,lawmaker,더불어민주당,국회 행정안전위원회,위원,NaN,박정현 의원(더불어민주당)은 박정현 의원은 행안위 첫 회의에 정부 관계자들이 불참한...
39,415회,배덕곤,unknown,NaN,NaN,소방청기획조정관,소방청기획조정관,배덕곤(소방청기획조정관)은 배덕곤은 R&D 예산이 기술 성숙도와 활용도 검증 결과 ...
40,415회,배준영,lawmaker,국민의힘,국회 행정안전위원회,위원,NaN,배준영 의원(국민의힘)은 배준영 의원은 여야 합의 정신을 존중하며 화성 공장 사고 ...
41,415회,신정훈,lawmaker,더불어민주당,국회 행정안전위원회,위원장,NaN,신정훈 의원(더불어민주당)은 신정훈 위원장은 행정안전부장관의 국회 출석 불참에 대해...
42,415회,양부남,lawmaker,더불어민주당,국회 행정안전위원회,위원,NaN,양부남 의원(더불어민주당)은 양부남 위원은 이태원참사 특별위원회 조속 구성과 이상민...


In [3]:
dup_by_round_summary = df[df.duplicated(
    subset=["회차", "발언자명", "발화내용 요약"],
    keep=False
)]

print(f"회차+요약 기준 중복 행 수: {len(dup_by_round_summary)}")
dup_by_round_summary.sort_values(["회차"]).head(10)


회차+요약 기준 중복 행 수: 0


,회차,발언자명,발언자유형,정당,소속기관,직위,소속_직위,발화내용 요약


In [5]:
df_dedup = df.drop_duplicates(
    subset=["회차","의원명", "발화내용 요약"],
    keep="first"
).reset_index(drop=True)

print(f"중복 제거 후 행 수: {len(df_dedup)}")


중복 제거 후 행 수: 571


In [6]:
df_dedup.to_csv(
    "회차별_발언자별_발화요약_소속정확_통합_dedup.csv",
    index=False,
    encoding="utf-8-sig"
)
